# День 6 — Inference pipeline и сохранение модели

**Цель:** пройти путь от исходного датасета до предсказаний на новых текстах без зависимости от артефактов предыдущих ноутбуков.

Ноутбук выполняется с нуля: сам загружает Financial PhraseBank, применяет общий preprocessing, обучает уже выбранную в день 4 конфигурацию, сохраняет весь pipeline и затем загружает его для inference.

## Почему мы не повторяем GridSearch

В день 4 конфигурация была выбрана cross-validation только на train-части, а в день 5 качество было исследовано на отложенной выборке. Повторный подбор по тем же данным не даст новой честной оценки. Для финального использования фиксируем победителя — unigram TF-IDF и LinearSVC — и обучаем его на всех доступных размеченных данных.

Метрики дней 4–5 относятся к модели, не видевшей test set. Финальная модель ниже обучена на всём датасете, поэтому мы не приписываем ей эти метрики.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import joblib
import pandas as pd
import sklearn
from IPython.display import display

from finnews_sentiment.data.load_data import DEFAULT_CONFIG, load_financial_phrasebank
from finnews_sentiment.features.preprocess import EXPECTED_SENTIMENTS, prepare_news_data
from finnews_sentiment.models.predict import load_model, predict_for_file, predict_texts
from finnews_sentiment.models.train_improved import build_day4_winner_pipeline

## 1. Пути и выходные файлы

Поиск корня проекта позволяет одинаково запускать ноутбук из корневой папки и из Jupyter. Каталоги создаются автоматически. Существующий файл модели, если он есть, не читается: ниже он будет полностью пересоздан.

In [2]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'pyproject.toml').exists() else current_dir.parent
assert (PROJECT_ROOT / 'pyproject.toml').exists(), 'Не найден корень проекта'

MODEL_PATH = PROJECT_ROOT / 'models' / 'best_model.joblib'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'day06'
DEMO_PATH = REPORT_DIR / 'demo_predictions.csv'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Модель: {MODEL_PATH.relative_to(PROJECT_ROOT)}')
print(f'Demo:   {DEMO_PATH.relative_to(PROJECT_ROOT)}')

Модель: models/best_model.joblib
Demo:   reports/day06/demo_predictions.csv


## 2. Загрузка и preprocessing с нуля

Источник — `financial_phrasebank`, конфигурация `sentences_75agree`. При первом запуске нужен доступ к Hugging Face, затем библиотека может использовать локальный кэш. `prepare_news_data` нормализует HTML, Unicode, регистр и пробелы, удаляет пустые строки и согласованные дубликаты. Числа, валюты, проценты и пунктуация сохраняются.

In [3]:
raw_df = load_financial_phrasebank(DEFAULT_CONFIG)
training_df = prepare_news_data(raw_df)

assert len(training_df) == 3448
assert training_df['text_clean'].ne('').all()
assert set(training_df['sentiment']) == set(EXPECTED_SENTIMENTS)
print(f'Исходных строк: {len(raw_df)}')
print(f'После preprocessing: {len(training_df)}')
display(training_df[['text', 'text_clean', 'sentiment']].head())

Исходных строк: 3453
После preprocessing: 3448


,text,text_clean,sentiment
0,"According to Gran , the company has no plans t...","according to gran , the company has no plans t...",neutral
1,With the new production plant the company woul...,with the new production plant the company woul...,positive
2,"For the last quarter of 2010 , Componenta 's n...","for the last quarter of 2010 , componenta 's n...",positive
3,"In the third quarter of 2010 , net sales incre...","in the third quarter of 2010 , net sales incre...",positive
4,Operating profit rose to EUR 13.1 mn from EUR ...,operating profit rose to eur 13.1 mn from eur ...,positive


## 3. Финальное обучение

Pipeline объединяет два шага: `TfidfVectorizer` сам превращает строки в числовые признаки, а `LinearSVC` выбирает класс. Благодаря этому vectorizer и классификатор нельзя случайно рассинхронизировать при сохранении или inference.

Параметры не подбираются заново: используется зафиксированный победитель дня 4 (`max_features=5000`, `min_df=2`, униграммы, `sublinear_tf=True`, `C=0.5`, балансировка классов).

In [4]:
final_model = build_day4_winner_pipeline(random_state=42)
final_model.fit(training_df['text_clean'], training_df['sentiment'])

print(final_model)
print(f"Размер словаря TF-IDF: {len(final_model.named_steps['tfidf'].vocabulary_)}")
print(f"Классы: {final_model.classes_.tolist()}")

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, min_df=2,
                                 sublinear_tf=True)),
                ('clf',
                 LinearSVC(C=0.5, class_weight='balanced', max_iter=5000,
                           random_state=42))])
Размер словаря TF-IDF: 3741
Классы: ['negative', 'neutral', 'positive']


## 4. Сохранение полного pipeline

В joblib-файл записываем сам обученный pipeline и metadata о его происхождении. Отдельный файл vectorizer не нужен: он уже находится в шаге `tfidf`.

> `joblib.load` использует механизм pickle и может выполнить вредоносный код. Загружайте только артефакты из доверенного источника. Совместимость также надёжнее при одинаковых версиях Python и scikit-learn.

In [5]:
metadata = {
    'artifact_version': 1,
    'task': 'financial_news_sentiment',
    'dataset': {
        'name': 'takala/financial_phrasebank',
        'config': DEFAULT_CONFIG,
    },
    'training_rows': len(training_df),
    'fit_scope': 'full_clean_dataset',
    'classes': list(final_model.classes_),
    'winner': 'linear_svc',
    'selection_origin': 'day04_cross_validation',
    'model_parameters': {
        'tfidf__max_features': 5000,
        'tfidf__min_df': 2,
        'tfidf__ngram_range': (1, 1),
        'tfidf__sublinear_tf': True,
        'clf__C': 0.5,
        'clf__class_weight': 'balanced',
    },
    'preprocessing': 'finnews_sentiment.features.preprocess.clean_text',
    'random_state': 42,
    'sklearn_version': sklearn.__version__,
}
joblib.dump({'model': final_model, 'meta': metadata}, MODEL_PATH)
print(f'Сохранено: {MODEL_PATH.relative_to(PROJECT_ROOT)} ({MODEL_PATH.stat().st_size:,} байт)')
display(pd.Series(metadata, name='value').to_frame())

Сохранено: models/best_model.joblib (228,516 байт)


,value
artifact_version,1
task,financial_news_sentiment
dataset,"{'name': 'takala/financial_phrasebank', 'confi..."
training_rows,3448
fit_scope,full_clean_dataset
classes,"[negative, neutral, positive]"
winner,linear_svc
selection_origin,day04_cross_validation
model_parameters,"{'tfidf__max_features': 5000, 'tfidf__min_df':..."
preprocessing,finnews_sentiment.features.preprocess.clean_text


## 5. Загрузка без переобучения

Удаляем ссылку на модель из памяти и загружаем созданный файл через публичную inference-функцию. Все последующие примеры используют только `loaded_model`: вызова `fit` больше не будет.

In [6]:
del final_model
loaded_model, loaded_metadata = load_model(MODEL_PATH)

assert loaded_metadata == metadata
assert hasattr(loaded_model.named_steps['tfidf'], 'vocabulary_')
print('Pipeline успешно загружен; повторное обучение не выполнялось.')

Pipeline успешно загружен; повторное обучение не выполнялось.


## 6. Предсказания для новых текстов

`predict_texts` принимает одну строку или последовательность строк. Результат содержит исходный и очищенный текст, класс и технические оценки LinearSVC. Чем больше `decision_margin` — разность двух лучших score, — тем дальше пример от спорной границы между двумя наиболее вероятными по решению классами. Это полезно для ранжирования примеров, но **score и margin не являются вероятностями**.

In [7]:
new_texts = [
    'The company reported strong profit growth and improved its outlook.',
    'The board will meet on Tuesday to review the quarterly report.',
    'Operating loss increased and the company issued a profit warning.',
    'Sales rose, but operating profit declined from the previous quarter.',
]
predictions = predict_texts(loaded_model, new_texts)
display(predictions.style.format({
    'score_negative': '{:.3f}',
    'score_neutral': '{:.3f}',
    'score_positive': '{:.3f}',
    'decision_margin': '{:.3f}',
}))

,text,text_clean,pred_sentiment,score_negative,score_neutral,score_positive,decision_margin
0,The company reported strong profit growth and improved its outlook.,the company reported strong profit growth and improved its outlook.,positive,-1.379,-0.968,1.187,2.155
1,The board will meet on Tuesday to review the quarterly report.,the board will meet on tuesday to review the quarterly report.,neutral,-1.047,0.474,-0.549,1.022
2,Operating loss increased and the company issued a profit warning.,operating loss increased and the company issued a profit warning.,negative,0.297,-1.163,-0.383,0.680
3,"Sales rose, but operating profit declined from the previous quarter.","sales rose, but operating profit declined from the previous quarter.",positive,-0.240,-0.729,-0.208,0.032


### Проверка единого preprocessing

Обучение использовало `text_clean`; inference обязан применять ту же функцию. Две записи ниже отличаются HTML, регистром, Unicode-сущностью и пробелами, но после очистки становятся одинаковыми и получают одинаковый результат.

In [8]:
normalization_demo = predict_texts(
    loaded_model,
    ['  <b>PROFIT&nbsp;rose</b> 10%  ', 'profit rose 10%'],
)
assert normalization_demo['text_clean'].nunique() == 1
assert normalization_demo['pred_sentiment'].nunique() == 1
display(normalization_demo[['text', 'text_clean', 'pred_sentiment']])

,text,text_clean,pred_sentiment
0,<b>PROFIT&nbsp;rose</b> 10%,profit rose 10%,positive
1,profit rose 10%,profit rose 10%,positive


## 7. CSV-вход и выход

Входной CSV обязан содержать колонку `text`; остальные колонки произвольны и сохраняются. Inference не удаляет строки и дубликаты, поэтому каждой входной строке соответствует ровно одна выходная. Пустые или нестроковые значения отклоняются явной ошибкой вместо скрытого предсказания по пустой строке.

In [9]:
demo_input = pd.DataFrame({
    'id': [101, 102, 103, 104],
    'source': ['wire', 'exchange', 'wire', 'wire'],
    'text': new_texts,
})

with TemporaryDirectory() as temporary_dir:
    temporary_input = Path(temporary_dir) / 'new_financial_news.csv'
    demo_input.to_csv(temporary_input, index=False)
    demo_predictions = predict_for_file(MODEL_PATH, temporary_input, DEMO_PATH)

assert len(demo_predictions) == len(demo_input)
assert demo_predictions['id'].tolist() == demo_input['id'].tolist()
print(f'Сохранено: {DEMO_PATH.relative_to(PROJECT_ROOT)}')
display(demo_predictions)

Сохранено: reports/day06/demo_predictions.csv


,id,source,text,text_clean,pred_sentiment,score_negative,score_neutral,score_positive,decision_margin
0,101,wire,The company reported strong profit growth and ...,the company reported strong profit growth and ...,positive,-1.379483,-0.967850,1.187036,2.154886
1,102,exchange,The board will meet on Tuesday to review the q...,the board will meet on tuesday to review the q...,neutral,-1.046600,0.473800,-0.548686,1.022486
2,103,wire,Operating loss increased and the company issue...,operating loss increased and the company issue...,negative,0.296993,-1.163419,-0.382913,0.679905
3,104,wire,"Sales rose, but operating profit declined from...","sales rose, but operating profit declined from...",positive,-0.240125,-0.729040,-0.208476,0.031649


## 8. Формат результата

Добавляемые колонки:

- `text_clean` — текст после того же preprocessing, что применялся при обучении;
- `pred_sentiment` — один из классов `negative`, `neutral`, `positive`;
- `score_<class>` — signed decision score LinearSVC для каждого класса;
- `decision_margin` — разность двух наибольших score.

Ограничения: модель обучена на коротких англоязычных финансовых фразах, не понимает длинный контекст как языковая модель, может путать косвенный sentiment с нейтральным сообщением и не поддерживает русский язык как заявленный домен.

## 9. Выводы

- Ноутбук воспроизводит весь путь от исходного датасета до сохранённого inference-артефакта.
- Один joblib-бандл содержит обученные TF-IDF и LinearSVC вместе с metadata.
- После загрузки новые тексты классифицируются без `fit` и без доступа к обучающим данным.
- Python API использует общий preprocessing и поддерживает одиночный и пакетный режимы.